# NB 6.6 &mdash; Projecte: el teu classificador

**MP 5149** &mdash; Desenvolupament de components software per a sistemes d'aprenentatge automàtic

**UT6** &mdash; Xarxes neuronals convolucionals i introducció a la visió per computador

---
### De què va

Al NB 6.4 vas posar a prova, amb la teva mà, un model entrenat amb mans generades per ordinador.
**Ara les fotos seran teves des del principi.** Entrenaràs un classificador amb objectes que triïs tu, en mesuraràs la
resistència, el desaràs per fer-lo servir a la UT8, i el faràs servir davant de la classe amb un objecte que no hagi
vist mai. Al final, una altra persona intentarà fer-lo fallar.

### Les regles

| Regla | Valor |
|---|---|
| **Classes** | 3 o 4 |
| **Fotos** | Un mínim de 30 per classe, fetes per tu |
| **Què es pot fotografiar** | **Només objectes** |
| **Què no** | **Cares, persones, matrícules ni dades privades** (documents, pantalles, adreces) |
| **Model** | ResNet18 + `MLPClassifier`, comparat amb una CNN des de zero. Si tens GPU, també pots afinar ResNet18 amb fine-tuning (pas 4 bis, opcional) |
| **Entrega** | Aquest notebook executat, amb la fitxa de resistència |

### Tria bé les fotos: és la part més important

Tria objectes que tinguis a mà i que es puguin confondre una mica: bolígraf, llapis i retolador; tres mena de fruita;
culleres, forquilles i ganivets.

**Canvia-ho tot entre foto i foto:** el fons, la llum, l'angle, la distància. Si fas les 30 fotos sobre la mateixa taula,
la xarxa pot aprendre la taula. És el que hem vist als mapes d'oclusió del NB 6.3. Procura que el fons no delati la classe:
no facis tots els bolígrafs a la cuina i tots els llapis a l'habitació.

## 0. Preparació

Les biblioteques i totes les eines que ja coneixes. **Aquesta part no cal modificar-la.**

In [ ]:
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from torch.utils.data import TensorDataset

plt.rcParams["figure.figsize"] = (6, 4)

La **plantilla d'entrenament**.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Plantilla d'entrenament. No cal modificar-la: només cal saber què fa cada funció.
# Fa el mateix que MLPClassifier feia per dins a la UT5, però a la vista.
# ─────────────────────────────────────────────────────────────────────────────
import math
import time

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

SEED = 42
LOSS_LIMIT = 100       # en un entrenament sa, la loss no s'hi acosta mai
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # GPU si n'hi ha; si no, CPU


def make_cnn(in_channels, n_classes, image_size=28, filters=(16, 32), pooling=True,
             global_pool=False, batch_norm=False, dropout=0.0):
    """Crea una CNN a partir de paràmetres, com MLPClassifier amb hidden_layer_sizes.

    Paràmetres:
        in_channels: canals de la imatge (1 en gris, 3 en color).
        n_classes: quantes classes ha de distingir.
        image_size: costat de la imatge en píxels.
        filters: filtres de cada capa convolucional; (16, 32) són dues capes.
        pooling: si cada capa redueix la imatge a la meitat amb MaxPool2d.
        global_pool: si al final es fa la mitjana de cada mapa en lloc d'aplanar-los.
        batch_norm: si s'afegeix BatchNorm2d després de cada convolució.
        dropout: fracció de neurones que s'apaguen a l'atzar a l'entrenament.

    Retorna el model. Exemple: model = make_cnn(1, 10, filters=(16, 32, 64))
    """
    if isinstance(filters, int):
        raise ValueError(f"filters ha de ser una llista de capes: per a una sola capa, escriu filters=({filters},), amb la coma.")
    if not len(filters) or min(filters) < 1:
        raise ValueError("filters necessita almenys una capa, amb un nombre positiu de filtres: per exemple, filters=(16, 32).")
    if not 0 <= dropout < 1:
        raise ValueError(f"dropout és la fracció de neurones que s'apaguen i ha d'anar de 0 a menys d'1: {dropout} no val. Prova dropout=0.5.")
    n_pools = (len(filters) - 1 if global_pool else len(filters)) if pooling else 0
    side = image_size // 2 ** n_pools
    if side < 1:
        raise ValueError(f"Hi ha massa capes amb pooling per a una imatge de {image_size} píxels: "
                         "fes filters més curt o prova pooling=False.")

    torch.manual_seed(SEED)
    layers = []
    channels = in_channels
    for position, n_filters in enumerate(filters):
        layers += [nn.Conv2d(channels, n_filters, kernel_size=3, padding=1)]
        if batch_norm:
            layers += [nn.BatchNorm2d(n_filters)]
        layers += [nn.ReLU()]
        is_last = position == len(filters) - 1
        if pooling and not (global_pool and is_last):
            layers += [nn.MaxPool2d(2)]
        channels = n_filters

    switch_off = [nn.Dropout(dropout)] if dropout > 0 else []
    if global_pool:
        layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), *switch_off, nn.Linear(channels, n_classes)]
    else:
        layers += [nn.Flatten(), nn.Linear(channels * side * side, 64), nn.ReLU(),
                   *switch_off, nn.Linear(64, n_classes)]
    return nn.Sequential(*layers)


def count_parameters(model):
    """Retorna quants pesos entrenables té el model."""
    return sum(weights.numel() for weights in model.parameters() if weights.requires_grad)


def make_loaders(train_set, val_set, batch_size=64, augment=None):
    """Prepara els mini-batches d'entrenament i de validació.

    Paràmetres:
        train_set, val_set: conjunts de dades de PyTorch.
        batch_size: imatges per mini-batch.
        augment: transformació per a cada imatge d'entrenament (data augmentation) o None.

    Retorna (train_loader, val_loader). Només l'entrenament es barreja i s'augmenta.
    """
    def collate_augmented(batch):
        images = torch.stack([augment(image) for image, _ in batch])
        labels = torch.as_tensor([label for _, label in batch])
        return images, labels

    collate = collate_augmented if augment is not None else None
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_set, batch_size=1000)
    return train_loader, val_loader


@torch.no_grad()
def _evaluate(model, loader):
    """Retorna la loss mitjana i l'encert del model amb un loader."""
    model.eval()
    device = next(model.parameters()).device
    loss_function = nn.CrossEntropyLoss(reduction="sum")
    total_loss, hits, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        scores = model(images)
        total_loss += loss_function(scores, labels).item()
        hits += (scores.argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, hits / total


def accuracy(model, loader):
    """Retorna l'encert del model (de 0 a 1), com score() a scikit-learn."""
    return _evaluate(model, loader)[1]


def train_model(model, train_loader, val_loader, epochs, lr, optimizer="adam"):
    """Entrena el model i mostra com va, epoch a epoch. És el fit() d'aquesta unitat.

    Paràmetres:
        model: la xarxa, feta amb make_cnn.
        train_loader, val_loader: els mini-batches, fets amb make_loaders.
        epochs: quantes vegades es recorren totes les dades d'entrenament.
        lr: el learning rate.
        optimizer: "adam" o "sgd".

    Retorna un diccionari amb les llistes train_loss, val_loss, train_acc i val_acc.
    """
    optimizers = {"adam": torch.optim.Adam, "sgd": torch.optim.SGD}
    if optimizer not in optimizers:
        raise ValueError(f"Optimitzador desconegut: {optimizer!r}. Tria 'adam' o 'sgd'.")

    torch.manual_seed(SEED)
    check_data(model, val_loader)
    model.to(DEVICE)
    step_maker = optimizers[optimizer](model.parameters(), lr=lr)
    loss_function = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        model.train()                                  # activa dropout i batch norm
        total_loss, hits, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            step_maker.zero_grad()                     # esborra el repartiment anterior
            scores = model(images)                     # 1. la xarxa prediu
            _check_labels(scores, labels)
            loss = loss_function(scores, labels)       # 2. mesurem l'error (la loss)
            loss.backward()                            # 3. backpropagation: repartim la culpa
            step_maker.step()                          # 4. una passa de gradient descent
            total_loss += loss.item() * len(labels)
            hits += (scores.argmax(dim=1) == labels).sum().item()
            total += len(labels)

        val_loss, val_acc = _evaluate(model, val_loader)
        history["train_loss"].append(total_loss / total)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(hits / total)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch}/{epochs} · loss {total_loss / total:.3f} · "
              f"encert a validació {val_acc:.1%} · {time.perf_counter() - start:.0f} s")

        if not math.isfinite(total_loss) or total_loss / total > LOSS_LIMIT:
            print("La loss s'ha disparat i l'entrenament s'atura: "
                  "segurament el learning rate és massa alt.")
            break
    return history


def check_data(model, loader):
    """Prova la xarxa amb una imatge del conjunt i, si no l'accepta, explica què cal canviar."""
    image, _ = loader.dataset[0]
    model.to(DEVICE)
    model.eval()
    try:
        with torch.no_grad():
            model(image.unsqueeze(0).to(DEVICE))
    except RuntimeError as error:
        channels, height, width = image.shape
        raise ValueError(f"La xarxa no accepta aquestes imatges, que tenen {channels} canals i fan {height} × {width} "
                         f"píxels. Crea-la de nou amb make_cnn({channels}, n_classes, image_size={height}), "
                         "amb la resta de paràmetres que ja tenies.") from error


def _check_labels(scores, labels):
    """Si hi ha més classes que sortides, la loss no es pot calcular: ho expliquem en lloc de fallar."""
    if labels.max() >= scores.shape[1]:
        raise ValueError(f"Aquest conjunt té etiquetes fins a la {labels.max().item()}, però la xarxa només té "
                         f"{scores.shape[1]} sortides. Crea-la de nou amb n_classes igual al nombre de classes: len(names).")


def plot_history(history):
    """Dibuixa les corbes de loss i d'encert d'un entrenament."""
    fig, grafics = plt.subplots(1, 2, figsize=(10, 3.5))
    epochs = range(1, len(history["train_loss"]) + 1)
    grafics[0].plot(epochs, history["train_loss"], "o-", label="entrenament")
    grafics[0].plot(epochs, history["val_loss"], "o-", label="validació")
    grafics[0].set_ylabel("loss")
    if not any(math.isfinite(value) for value in history["train_loss"] + history["val_loss"]):
        grafics[0].text(0.5, 0.5, "La loss s'ha disparat:\nprova un learning rate més petit.",
                        transform=grafics[0].transAxes, ha="center", va="center")
    grafics[1].plot(epochs, history["train_acc"], "o-", label="entrenament")
    grafics[1].plot(epochs, history["val_acc"], "o-", label="validació")
    grafics[1].set_ylabel("encert")
    for grafic in grafics:
        grafic.set_xlabel("epoch")
        grafic.legend()
    plt.tight_layout()
    plt.show()

Les eines del NB 6.4 (ResNet18 i fotos) i del NB 6.3 (crash test), i les noves del projecte.

In [ ]:
# Funcions per reaprofitar una xarxa ja entrenada i carregar fotos. No cal modificar-la.
import io
import zipfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageOps, UnidentifiedImageError
from torch import nn
from torchvision import models
from torchvision.transforms import v2

WEIGHTS = models.ResNet18_Weights.IMAGENET1K_V1
_LOADED = {}


def _resnet():
    """Carrega ResNet18 una sola vegada: la primera crida baixa uns 45 MB de pesos."""
    if not _LOADED:
        model = models.resnet18(weights=WEIGHTS).eval()
        _LOADED["model"] = model
        _LOADED["backbone"] = nn.Sequential(*list(model.children())[:-1], nn.Flatten()).eval()
        _LOADED["prepare"] = WEIGHTS.transforms()
    return _LOADED


def _to_rgb(images):
    """Converteix imatges en gris a tres canals, que és el que espera ResNet18."""
    return images.repeat_interleave(3, dim=-3) if images.shape[-3] == 1 else images


@torch.no_grad()
def top3_imagenet(image):
    """Retorna les tres classes d'ImageNet més probables d'una imatge: [(nom en anglès, probabilitat), ...]."""
    resnet = _resnet()
    scores = resnet["model"](resnet["prepare"](_to_rgb(image).unsqueeze(0)))[0]
    values, positions = torch.softmax(scores, dim=0).topk(3)
    return [(WEIGHTS.meta["categories"][p], v) for p, v in zip(positions.tolist(), values.tolist())]


@torch.no_grad()
def extract_features(images, batch_size=32):
    """Passa les imatges per ResNet18, sense l'última capa, i retorna 512 números per imatge."""
    resnet = _resnet()
    pieces = [resnet["backbone"](resnet["prepare"](_to_rgb(batch)))
              for batch in torch.split(images, batch_size)]
    return torch.cat(pieces).numpy()


def _load_rgb(file, size):
    """Obre una imatge d'un fitxer, respecta l'orientació del mòbil i la passa a tensor RGB."""
    image = ImageOps.exif_transpose(Image.open(file)).convert("RGB")
    if size is not None:
        image = image.resize((size, size), Image.BILINEAR)
    return torch.tensor(np.array(image) / 255.0, dtype=torch.float32).permute(2, 0, 1)


def load_image_zip(url_or_path, size=224):
    """Carrega un zip amb carpetes train/<classe>/ i test/<classe>/.

    Retorna un diccionari amb les claus train i test: cadascuna (imatges, etiquetes, noms de classe).
    """
    if str(url_or_path).startswith(("http", "file:")):
        with urllib.request.urlopen(str(url_or_path), timeout=300) as response:
            content = response.read()
    else:
        content = Path(url_or_path).read_bytes()
    with zipfile.ZipFile(io.BytesIO(content)) as archive:
        files = sorted(name for name in archive.namelist() if name.lower().endswith((".jpg", ".jpeg", ".png")))
        class_names = sorted({name.split("/")[1] for name in files})
        data = {}
        for part in ["train", "test"]:
            chosen = [name for name in files if name.startswith(part + "/")]
            images = torch.stack([_load_rgb(io.BytesIO(archive.read(name)), size) for name in chosen])
            labels = torch.tensor([class_names.index(name.split("/")[1]) for name in chosen])
            data[part] = (images, labels, class_names)
    return data


def load_photo(path, size=224):
    """Carrega una foto com a tensor (3, alt, ample) amb valors de 0 a 1.

    Paràmetres:
        path: on és el fitxer.
        size: el costat de la imatge quadrada que es vol. None la deixa com és.
    """
    if path is None:
        raise ValueError("No hi ha cap foto: torna a executar la cel·la i tria'n una, o escriu-ne la ruta: load_photo('foto.jpg').")
    try:
        return _load_rgb(path, size)
    except FileNotFoundError as error:
        raise ValueError(f"No trobo {path}: comprova que l'has pujat i que el nom és correcte.") from error
    except UnidentifiedImageError as error:
        raise ValueError(f"No es pot obrir {Path(path).name}. Si és una foto HEIC d'iPhone, converteix-la "
                         "a JPG: obre-la a l'ordinador i exporta-la, o canvia la càmera a «Més compatible».") from error


def upload_photo():
    """Obre el selector de fitxers de Colab i retorna el nom de la foto, o None si no n'hi ha."""
    try:
        from google.colab import files
    except ImportError:
        print("Aquest botó només funciona a Colab. Aquí pots fer servir load_photo('ruta/de/la/foto.jpg').")
        return None
    uploaded = files.upload()
    return next(iter(uploaded), None)


def take_per_class(labels, n, seed=42):
    """Tria n imatges de cada classe a l'atzar i en retorna les posicions."""
    generator = torch.Generator().manual_seed(seed)
    chosen = []
    for label in labels.unique():
        positions = (labels == label).nonzero().flatten()
        if len(positions) < n:
            raise ValueError(f"La classe {label.item()} només té {len(positions)} imatges: tria un n més petit.")
        chosen.append(positions[torch.randperm(len(positions), generator=generator)[:n]])
    return torch.cat(chosen)


def predict_photo(mlp, image, class_names):
    """Passa una foto per ResNet18 i pel teu MLPClassifier. Retorna [(nom, probabilitat), ...], de més a menys."""
    probabilities = mlp.predict_proba(extract_features(image.unsqueeze(0)))[0]
    return sorted(zip(class_names, probabilities.tolist()), key=lambda pair: -pair[1])


def show_prediction(image, top):
    """Dibuixa una foto al costat de les probabilitats que hi dona el model."""
    fig, grafics = plt.subplots(1, 2, figsize=(7.5, 2.8))
    grafics[0].imshow(image.permute(1, 2, 0).squeeze().numpy(), cmap="gray")
    grafics[0].axis("off")
    grafics[1].barh([name for name, _ in top][::-1], [value for _, value in top][::-1], color="#1C7293")
    grafics[1].set_xlim(0, 1)
    grafics[1].set_xlabel("probabilitat")
    plt.tight_layout()
    plt.show()


def make_pretrained(n_classes, unfreeze=True):
    """Crea una ResNet18 amb una última capa nova, per fer fine-tuning amb train_model.

    Paràmetres:
        n_classes: quantes classes ha de distingir.
        unfreeze: si també es deixa aprendre l'últim bloc de capes (layer4).

    Va millor amb GPU i imatges de 224 × 224. Sense GPU, amb imatges de 112 × 112 també es pot provar.
    """
    torch.manual_seed(SEED)
    network = models.resnet18(weights=WEIGHTS)
    for weights in network.parameters():
        weights.requires_grad = False
    if unfreeze:
        for weights in network.layer4.parameters():
            weights.requires_grad = True
    network.fc = nn.Linear(network.fc.in_features, n_classes)
    return nn.Sequential(v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), network)


@torch.no_grad()
def predict_finetuned(model, image, class_names):
    """Passa una foto per una xarxa feta amb make_pretrained. Retorna [(nom, probabilitat), ...], de més a menys.

    La foto ha de tenir la mateixa mida que les imatges amb què l'has entrenada: load_photo(path, size=size).
    """
    model.eval()
    device = next(model.parameters()).device
    probabilities = torch.softmax(model(_to_rgb(image).unsqueeze(0).to(device))[0], dim=0).cpu()
    return sorted(zip(class_names, probabilities.tolist()), key=lambda pair: -pair[1])


# Funcions per posar a prova una xarxa. No cal modificar-la.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision.transforms import v2

DEFAULT_LEVELS = {"noise": [0, 0.05, 0.1, 0.2, 0.3],        # desviació del soroll (píxels de 0 a 1)
                  "blur": [0, 0.5, 1.0, 1.5, 2.0],           # força del difuminat
                  "rotation": [0, 10, 20, 30, 45],           # graus
                  "brightness": [0, 0.1, 0.2, 0.3, 0.5],     # quant s'aclareix
                  "occlusion": [0, 0.125, 0.25, 0.375, 0.5]}  # costat del tros tapat, en fracció de la imatge
CRASH_NAMES = {"noise": "soroll", "blur": "difuminat", "rotation": "gir",
               "brightness": "brillantor", "occlusion": "tros tapat"}


def load_model(name):
    """Carrega una de les dues CNN de CIFAR-10 ja entrenades: "cifar_cnn" o "cifar_cnn_aug"."""
    if name not in ("cifar_cnn", "cifar_cnn_aug"):
        raise ValueError(f"Model desconegut: {name!r}. Els que hi ha: cifar_cnn, cifar_cnn_aug.")
    model = make_cnn(3, 10, image_size=32, filters=(16, 32, 64), batch_norm=True)
    file = _download(f"{URL_BASE}UT06-Xarxes_convolucionals/models/{name}.pt")
    model.load_state_dict(torch.load(file, weights_only=True, map_location="cpu"))
    return model.eval()


def corrupt(images, kind, level, generator):
    """Espatlla un lot d'imatges amb la prova i la intensitat indicades."""
    if level == 0:
        return images
    if kind == "noise":
        return (images + level * torch.randn(images.shape, generator=generator)).clamp(0, 1)
    if kind == "blur":
        return v2.functional.gaussian_blur(images, kernel_size=2 * int(3 * level) + 1, sigma=float(level))
    if kind == "rotation":
        return v2.functional.rotate(images, float(level))
    if kind == "brightness":
        return (images + level).clamp(0, 1)
    side = images.shape[-1]
    patch = max(1, round(level * side))
    tops = torch.randint(0, side - patch + 1, (len(images),), generator=generator).tolist()
    lefts = torch.randint(0, side - patch + 1, (len(images),), generator=generator).tolist()
    covered = images.clone()
    for position, (top, left) in enumerate(zip(tops, lefts)):
        covered[position, :, top:top + patch, left:left + patch] = 0.5
    return covered


@torch.no_grad()
def crash_test(model, loader, kind, levels=None):
    """Mesura l'encert del model mentre s'espatllen les imatges cada vegada més.

    Paràmetres:
        model: la xarxa que es prova.
        loader: les imatges de prova.
        kind: "noise", "blur", "rotation", "brightness" o "occlusion".
        levels: les intensitats. Si no se'n donen, es fan servir unes per defecte.

    Retorna una taula amb les columnes intensitat i encert.
    """
    if kind not in DEFAULT_LEVELS:
        raise ValueError(f"Prova desconeguda: {kind!r}. Tria'n una d'aquestes: {', '.join(DEFAULT_LEVELS)}.")
    model.eval()
    device = next(model.parameters()).device
    rows = []
    for level in (DEFAULT_LEVELS[kind] if levels is None else levels):
        generator = torch.Generator().manual_seed(SEED)
        hits, total = 0, 0
        for images, labels in loader:
            guesses = model(corrupt(images, kind, level, generator).to(device)).argmax(dim=1).cpu()
            hits += (guesses == labels).sum().item()
            total += len(labels)
        rows.append({"intensitat": level, "encert": hits / total})
    return pd.DataFrame(rows)


def plot_crash(tables, title=""):
    """Dibuixa corbes de resistència. tables és un diccionari {nom del model: taula de crash_test}."""
    fig, grafic = plt.subplots()
    for label, table in tables.items():
        grafic.plot(table["intensitat"], table["encert"], "o-", label=label)
    grafic.set_xlabel("intensitat")
    grafic.set_ylabel("encert")
    grafic.set_ylim(0, 1)
    grafic.set_title(title)
    grafic.legend()
    plt.show()
    return grafic


def resistance_card(model, loader):
    """Fa totes les proves i les resumeix en una fitxa de resistència."""
    rows = {}
    for kind, name in CRASH_NAMES.items():
        table = crash_test(model, loader, kind)
        clean, worst = table["encert"].iloc[0], table["encert"].iloc[-1]
        rows[name] = {"encert sense tocar": round(clean, 3),
                      "encert a la intensitat més forta": round(worst, 3),
                      "pèrdua (punts)": round(100 * (clean - worst), 1)}
    return pd.DataFrame(rows).T


def show_corruptions(image, kind):
    """Dibuixa una imatge amb totes les intensitats per defecte d'una prova."""
    levels = DEFAULT_LEVELS[kind]
    fig, grafics = plt.subplots(1, len(levels), figsize=(1.5 * len(levels), 1.9))
    generator = torch.Generator().manual_seed(SEED)
    for level, grafic in zip(levels, grafics):
        changed = corrupt(image.unsqueeze(0), kind, level, generator)[0]
        grafic.imshow(changed.permute(1, 2, 0).squeeze().numpy(), cmap="gray")
        grafic.set_title(str(level), fontsize=9)
        grafic.axis("off")
    fig.suptitle(f"{CRASH_NAMES[kind]}: intensitats", y=1.02)
    plt.show()


@torch.no_grad()
def occlusion_map(model, image, patch=6):
    """Tapa cada zona de la imatge i mesura quant baixa la confiança en la classe predita.

    Retorna una graella: com més alt el valor, més importava aquella zona per decidir.
    """
    model.eval()
    device = next(model.parameters()).device
    image = image.to(device)
    probabilities = torch.softmax(model(image.unsqueeze(0))[0], dim=0)
    chosen = probabilities.argmax()
    fill = image.mean(dim=(1, 2), keepdim=True)
    starts = range(0, image.shape[-1] - patch + 1, 2)
    result = np.zeros((len(starts), len(starts)))
    for row, top in enumerate(starts):
        covered = image.repeat(len(starts), 1, 1, 1)
        for col, left in enumerate(starts):
            covered[col, :, top:top + patch, left:left + patch] = fill
        result[row] = (probabilities[chosen] - torch.softmax(model(covered), dim=1)[:, chosen]).cpu().numpy()
    return result


def show_occlusion(model, image, class_names, patch=6):
    """Dibuixa la imatge i el seu mapa d'oclusió, amb la classe predita."""
    result = occlusion_map(model, image, patch)
    model.eval()
    with torch.no_grad():
        scores = model(image.unsqueeze(0).to(next(model.parameters()).device))[0]
    picture = image.permute(1, 2, 0).squeeze().cpu().numpy()
    fig, grafics = plt.subplots(1, 2, figsize=(6.5, 3))
    grafics[0].imshow(picture, cmap="gray")
    grafics[0].set_title(f"diu: {class_names[int(scores.argmax())]}")
    grafics[1].imshow(picture, cmap="gray", alpha=0.35)
    side = image.shape[-1]
    limit = max(abs(result).max(), 1e-6)
    grafics[1].imshow(result, cmap="RdBu_r", vmin=-limit, vmax=limit, alpha=0.75, extent=(0, side, side, 0))
    grafics[1].set_title("vermell: on mira")
    for grafic in grafics:
        grafic.axis("off")
    plt.show()


def attack(model, image, force):
    """Canvia cada píxel, com a molt, force, en la direcció que més fa dubtar el model.

    És la backpropagation al revés: en lloc de canviar els pesos per reduir la loss,
    es canvien els píxels per augmentar-la. Retorna la imatge atacada.
    """
    if force == 0:
        return image.clone()
    model.eval()
    device = next(model.parameters()).device
    with torch.enable_grad():
        pixels = image.to(device).unsqueeze(0).clone().requires_grad_(True)
        scores = model(pixels)
        loss = nn.functional.cross_entropy(scores, scores.argmax(dim=1))
        direction, = torch.autograd.grad(loss, pixels)
    return (pixels + force * direction.sign()).clamp(0, 1)[0].detach().cpu()


def attacked_accuracy(model, dataset, force, n=200):
    """Retorna l'encert del model amb les primeres n imatges, atacades una a una."""
    model.eval()
    device = next(model.parameters()).device
    hits = 0
    for position in range(min(n, len(dataset))):
        image, label = dataset[position]
        with torch.no_grad():
            guess = model(attack(model, image, force).unsqueeze(0).to(device)).argmax(dim=1).item()
        hits += guess == int(label)
    return hits / min(n, len(dataset))


@torch.no_grad()
def show_attack(model, image, force, class_names):
    """Dibuixa l'original, el canvi (molt augmentat) i la imatge atacada, amb què diu el model."""
    attacked = attack(model, image, force)
    device = next(model.parameters()).device
    model.eval()
    fig, grafics = plt.subplots(1, 3, figsize=(7.5, 2.9))
    for picture, grafic, label in [(image, grafics[0], "original"), (attacked, grafics[2], "atacada")]:
        probabilities = torch.softmax(model(picture.unsqueeze(0).to(device))[0], dim=0)
        grafic.imshow(picture.permute(1, 2, 0).squeeze().cpu().numpy(), cmap="gray")
        grafic.set_title(f"{label}: {class_names[int(probabilities.argmax())]} ({probabilities.max().item():.0%})", fontsize=10)
    change = (attacked - image.cpu()).permute(1, 2, 0).squeeze().numpy()
    limit = max(abs(change).max(), 1e-6)
    grafics[1].imshow((change / limit + 1) / 2, cmap="gray")
    grafics[1].set_title(f"el canvi, augmentat (força {force})", fontsize=10)
    for grafic in grafics:
        grafic.axis("off")
    plt.show()


# Funcions per carregar les teves fotos i mesurar el model. No cal modificar-la.
import io
import zipfile
from pathlib import PurePosixPath

import pandas as pd
import torch
from PIL import UnidentifiedImageError

MIN_PHOTOS = 30
PHOTO_ENDINGS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")


def _is_junk(name):
    """Els fitxers que el sistema operatiu afegeix als zips no compten per a res."""
    parts = PurePosixPath(name).parts
    return name.endswith("/") or "__MACOSX" in parts or parts[-1].startswith(".") or parts[-1] == "Thumbs.db"


def load_photos_zip(path, size=224):
    """Carrega un zip amb una carpeta per classe: boli/1.jpg, boli/2.jpg, goma/1.jpg…

    Retorna (imatges, etiquetes, noms de les classes, descartats). Descartats és una llista de
    (fitxer, motiu) amb tot el que no s'ha pogut llegir.
    """
    if path is None:
        raise ValueError("No hi ha cap zip: torna a executar la cel·la i tria'n un, o escriu-ne la ruta: load_photos_zip('fotos.zip').")
    try:
        archive = zipfile.ZipFile(path)
    except FileNotFoundError as error:
        raise ValueError(f"No trobo {path}: comprova que l'has pujat i que el nom és correcte.") from error
    except zipfile.BadZipFile as error:
        raise ValueError(f"{path} no és un fitxer zip vàlid: torna a comprimir la carpeta de les fotos.") from error
    structure = "Cal una carpeta per classe dins del zip, com ara boli/1.jpg, boli/2.jpg, goma/1.jpg…"
    with archive:
        files = sorted(name for name in archive.namelist() if not _is_junk(name))
        roots = {PurePosixPath(name).parts[0] for name in files}
        if len(roots) == 1 and all(len(PurePosixPath(name).parts) >= 3 for name in files):
            depth = 1      # una carpeta que ho embolcalla tot, com passa en comprimir-la
        else:
            depth = 0
        if any(len(PurePosixPath(name).parts) < depth + 2 for name in files) or not files:
            raise ValueError("Hi ha fotos fora de les carpetes. " + structure)

        found, skipped = {}, []
        for name in files:
            class_name = PurePosixPath(name).parts[depth]
            if name.lower().endswith((".heic", ".heif")):
                skipped.append((name, "Format HEIC d'iPhone: converteix-la a JPG, o tria «Més compatible» a la càmera."))
            elif not name.lower().endswith(PHOTO_ENDINGS):
                skipped.append((name, "No és una imatge."))
            else:
                try:
                    found.setdefault(class_name, []).append(_load_rgb(io.BytesIO(archive.read(name)), size))
                except (UnidentifiedImageError, OSError):
                    skipped.append((name, "No s'ha pogut obrir: pot ser que estigui trencada."))

    if len(found) < 2:
        reasons = " ".join(sorted({reason for _, reason in skipped}))
        detail = f"S'han descartat {len(skipped)} fitxers. {reasons}" if skipped else structure
        raise ValueError(f"Classes amb fotos que es poden llegir: {len(found)}. En calen 3 o 4. " + detail)
    class_names = sorted(found)
    images = torch.stack([image for name in class_names for image in found[name]])
    labels = torch.tensor([number for number, name in enumerate(class_names) for _ in found[name]])
    counts = {name: len(found[name]) for name in class_names}
    print("Fotos per classe:", counts)
    low = {name: n for name, n in counts.items() if n < MIN_PHOTOS}
    if low:
        print(f"Avís: poques fotos a {low}. El mínim del projecte és de {MIN_PHOTOS} per classe.")
    if skipped:
        print(f"S'han descartat {len(skipped)} fitxers: mira la llista de descartats.")
    return images, labels, class_names, skipped


def project_card(mlp, images, labels):
    """Fa la fitxa de resistència del model ResNet18 + MLPClassifier.

    Espatlla les imatges a la intensitat més forta de cada prova i torna a mesurar l'encert.
    """
    target = labels.numpy()
    clean = mlp.score(extract_features(images), target)
    rows = {}
    for kind, name in CRASH_NAMES.items():
        generator = torch.Generator().manual_seed(SEED)
        worst = mlp.score(extract_features(corrupt(images, kind, DEFAULT_LEVELS[kind][-1], generator)), target)
        rows[name] = {"encert sense tocar": round(clean, 3),
                      "encert a la intensitat més forta": round(worst, 3),
                      "pèrdua (punts)": round(100 * (clean - worst), 1)}
    return pd.DataFrame(rows).T

A partir d'aquí, **la feina és teva**. Cada pas t'explica què has de fer i et dona el codi de partida, que
hauràs d'adaptar.

## Pas 1. Puja les fotos

Fes una carpeta per classe, amb el nom de la classe, i comprimeix-les en un zip:

```
boli/foto1.jpg
boli/foto2.jpg
goma/foto1.jpg
llapis/foto1.jpg
```

Puja'l i carrega'l. `load_photos_zip` t'avisarà si n'hi ha alguna que no pot llegir o si et falten fotos. Si fas servir un
iPhone, les fotos poden ser HEIC: la funció t'ho dirà, i les hauràs de convertir a JPG.

```python
zip_name = upload_photo()          # fora de Colab, escriu directament la ruta del zip
images, labels, class_names, skipped = load_photos_zip(zip_name)
print(class_names, tuple(images.shape))
print(skipped)
```

Dibuixa'n unes quantes per comprovar que s'han carregat bé.

In [ ]:
# El teu codi aquí

## Pas 2. Separa'n una part per provar

Una quarta part de les fotos no s'ha de fer servir per entrenar. El paràmetre `stratify` fa que cada classe tingui
la mateixa proporció a totes dues parts: amb 30 fotos per classe, n'hi queden unes 8 de cada per provar. Sense `stratify`,
podria ser que quedessin 4 d'una classe i 12 d'una altra.

Separem **posicions**, no imatges, perquè així podrem provar exactament les mateixes fotos amb tots dos models.

```python
train_positions, test_positions = train_test_split(np.arange(len(labels)), test_size=0.25,
                                                   stratify=labels.numpy(), random_state=SEED)
```

**Quin és l'encert de qui endevina a l'atzar?** Amb 3 classes, un 33%. Aquesta és la primera referència: si no el superes,
el teu model no ha après res.

In [ ]:
# El teu codi aquí

## Pas 3. El model de referència: una CNN des de zero

Abans de fer servir res sofisticat, cal saber què s'aconsegueix amb el més senzill. Entrena una CNN des de zero, com
al NB 6.4, amb les fotos a 64 × 64 per anar de pressa, i apunta'n l'encert.

```python
small_images = load_photos_zip(zip_name, size=64)[0]
loader, test_loader = make_loaders(TensorDataset(small_images[train_positions], labels[train_positions]),
                                   TensorDataset(small_images[test_positions], labels[test_positions]), batch_size=16)
reference = make_cnn(3, len(class_names), image_size=64, filters=(16, 32, 64), batch_norm=True)
history = train_model(reference, loader, test_loader, epochs=20, lr=0.001)
```

In [ ]:
# El teu codi aquí

## Pas 4. El teu model: ResNet18 + `MLPClassifier`

Exactament el mateix del NB 6.4, però amb les teves fotos.

```python
features = extract_features(images)
mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=SEED)
mlp.fit(features[train_positions], labels[train_positions].numpy())
print("Encert de test:", mlp.score(features[test_positions], labels[test_positions].numpy()))
```

Compara'l amb el pas 3 i amb l'atzar. Mira quines fotos falla i amb quina classe les confon: `mlp.predict` et dona les
respostes, i les pots dibuixar. **Si n'hi ha alguna que sembli una errada teva, com una foto mal etiquetada, torna al pas 1.**

In [ ]:
# El teu codi aquí

## Pas 4 bis (opcional). Afina ResNet18

Si tens GPU, prova el **fine-tuning** de la missió 6 del NB 6.4 amb les teves fotos. Fes-lo amb les mateixes posicions
de test, perquè la comparació amb el pas 3 i el pas 4 sigui justa. `make_pretrained` prepara la xarxa i `train_model`
l'entrena, com sempre.

```python
size = 224 if DEVICE == "cuda" else 112      # sense GPU, imatges més petites
ft_images = load_photos_zip(zip_name, size=size)[0]
loader, test_loader = make_loaders(TensorDataset(ft_images[train_positions], labels[train_positions]),
                                   TensorDataset(ft_images[test_positions], labels[test_positions]), batch_size=16)
tuned = make_pretrained(len(class_names), unfreeze=True)
history = train_model(tuned, loader, test_loader, epochs=5, lr=0.0001)
```

Supera l'MLP? Val la pena el temps de més? Al pas 6 i al pas 8 el pots provar al costat de l'MLP amb
`predict_finetuned(tuned, photo, class_names)`, que funciona com `predict_photo`. La fitxa del pas 5 i el model que
desaràs al pas 7 continuen sent els de l'MLP: és el que tornaràs a carregar a la UT8.

> ⏱ Amb GPU, uns segons. Sense GPU, amb un centenar de fotos de 112 × 112, uns 10 segons a un ordinador normal i el
> triple a Colab.

In [ ]:
# El teu codi aquí

## Pas 5. La fitxa de resistència

`project_card` és la fitxa del NB 6.3, ara per a aquest model. Fa les cinc proves a la intensitat més forta.

```python
card = project_card(mlp, images[test_positions], labels[test_positions])
card
```

Escriu sota la fitxa, en tres frases i sense paraules tècniques, què pot fer el teu model i en quines condicions no
te'n pots refiar. **Aquesta part es valora molt.**

In [ ]:
# El teu codi aquí

## Pas 6. Fotos noves

Les fotos que has fet servir fins ara les vas fer el mateix dia i al mateix lloc. **Fes-ne 6 de noves**, si pot ser
en un altre lloc i amb una altra llum, i prova-les una a una. Apunta els resultats en una taula.

```python
photo = load_photo(upload_photo())
show_prediction(photo, predict_photo(mlp, photo, class_names))
```

L'encert és semblant al del pas 4? Si és molt més baix, per què creus que passa?

In [ ]:
# El teu codi aquí

## Pas 7. Desa el model

Un model que no es pot tornar a carregar no serveix de gaire. **`joblib`** desa objectes de Python en un fitxer. Desa
l'MLP i els noms de les classes, que formen part del model. **A la UT8 el tornaràs a carregar** per fer-ne servir les
prediccions, sense tornar a entrenar.

```python
joblib.dump({"mlp": mlp, "class_names": class_names, "features": "ResNet18 IMAGENET1K_V1"}, "el_meu_model.joblib")
saved = joblib.load("el_meu_model.joblib")
print(saved["class_names"])
```

A Colab, els fitxers desapareixen en acabar la sessió. **Baixa'l** des de la carpeta del panell lateral, o amb
`from google.colab import files; files.download("el_meu_model.joblib")`.

In [ ]:
# El teu codi aquí

## Pas 8. El repte i la demo

**Repte.** Intercanvia't amb una altra persona que tingui classes semblants. Cadascú fa **10 fotos dels objectes de
l'altra persona, intentant que el model falli**: amb un fons estrany, amb una ombra o molt a prop, però sense
fer trampes (la classe s'ha de poder reconèixer). Apunta quantes en encerta el teu model. Si les fotos són en un zip amb la
mateixa estructura, pots fer-ho de cop:

```python
other_images, other_labels, other_names, _ = load_photos_zip("fotos_de_l_altra_persona.zip")
print(mlp.score(extract_features(other_images), other_labels.numpy()))
```

Comprova que `other_names` coincideix amb `class_names`: les etiquetes es compten per ordre alfabètic.

**Demo.** Cinc minuts davant de la classe:

1. Quines classes has triat, i com has fet les fotos.
2. Quin encert has mesurat, i la fitxa de resistència.
3. Prova en directe, amb **un objecte que el model no hagi vist mai**.
4. Què hauries de canviar perquè funcionés de debò.

In [ ]:
# El teu codi aquí

## Com s'avaluarà

| Apartat | Excel·lent | Suficient | Insuficient |
|---|---|---|---|
| **Dades** | 3–4 classes, més de 30 fotos per classe i molta varietat de fons, llum i angle. Sense res privat | Mínim complert, amb poca varietat | Menys de 30 fotos, o surten persones o dades privades |
| **Procés** | Tots els passos, amb les mateixes fotos de prova per als dos models, i el model desat | Tots els passos, amb algun error petit | Falten passos, o s'ha entrenat amb les fotos de prova |
| **Interpretació** | Explica què aguanta i què no, per què falla i què canviaria, amb números | Comenta els resultats, sense gaire justificació | Només dona l'encert |
| **Demo** | Clara, en 5 minuts i amb una prova en directe que sap explicar, encara que falli | Es fa, però no s'explica bé el resultat | No es fa |

**Un model que falla i que saps explicar val més que un model que encerta i no saps per què.**

## Entrega

Lliura per Google Classroom, en la data que t'indiqui el professorat:

- Aquest notebook **executat**, amb les taules i els gràfics a la vista.
- El fitxer `el_meu_model.joblib`.
- Si no et cap al Classroom, no cal lliurar-hi les fotos, però guarda-les: te les poden demanar.